# Supplementary Figure 15 | Cross-tissue CoS composition and lead enrichment

**a.** Co-occurrence of QTL biotypes among single-tissue GWAS-QTL CoS (GWAS p < 10-5), concatenated across tissues. Rows are lncRNA eQTL, lncRNA sQTL, protein-coding eQTL, and protein-coding sQTL. Each cell is the percentage of CoS with that row biotype that also contain the column biotype. A CoS is counted in every column it matches, so a row can sum to more than 100. Other gene is CoS that contain a biotype other than protein-coding or lncRNA (i.e., pseudogene). Alone is CoS that contain only a single phenotype. Color, % of single-tissue GWAS-QTL CoS. Black outline, cells that count as multi-gene.

**b.** Percentage of GWAS-QTL CoS (GWAS p < 10-5) that are multi-gene, by biotype. A CoS is multi-gene when it contains QTLs from two or more genes. Dots are per-tissue single-tissue rates, colored by GTEx tissue. Outline bars are cross-tissue rate, with the edge colored by biotype.

**c.** Flow of single-tissue GWAS-QTL CoS (GWAS p < 10-5) into their cross-tissue locus for CoS that contain a protein-coding gene (left) and CoS that contain a lncRNA (right). A CoS with both is counted in both. Some CoS are single-gene in one tissue, but multi-gene in the merged cross-tissue locus.

**d.** Distribution of the number of tissues with a QTL in the CoS per cross-tissue GWAS-QTL CoS (GWAS p < 10-5).

**e.** Distribution of the number of GWAS per cross-tissue GWAS-QTL CoS (GWAS p < 10-5).

**f.** Distribution of the number of genes with a QTL in the CoS per cross-tissue GWAS-QTL CoS (GWAS p < 10-5).

**g.** Distribution of the number of QTL phenotypes per cross-tissue GWAS-QTL CoS.

**h.** VEP annotation enrichment of GWAS-QTL CoS (GWAS p < 5.7 × 10-9) lead variants from cross-tissue CoS, stratified by cross-tissue GWAS-QTL CoS composition (lncRNA only, n = 397; protein-coding only, n = 2,921; mixed, n = 4,222), versus non-colocalized variants in the cis windows of the corresponding biotype(s), with allele frequency as a covariate. Points, logistic-regression odds ratios; error bars, 95% CIs; y-axis, odds ratio on a log2 scale, sorted by the lncRNA-only odds ratio. Asterisks, two-sided Wald p (*, p < 0.05; **, p < 0.01; ***, p < 0.001; ****, p < 0.0001).


## Setup

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from tqdm.auto import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

gene_type_palette = {
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "pseudogene": sns.color_palette("Paired", 12)[3],
    "sRNA": sns.color_palette("Paired", 12)[7],
}

gwas_p_cut_1e5 = 1e-5
gwas_neglog10p_1e5 = -np.log10(gwas_p_cut_1e5)

_ensg_stem = re.compile(r"(ENSG\d+)")
qtl_label_for_figure = {
    "no_qtl": "no_qtl",
    "non-lncRNA": "non-lncRNA",
    "Shared": "Shared",
    "lncRNA-specific (weak)": "Shared",
    "lncRNA-specific (robust)": "lncRNA-specific",
}

_paired = sns.color_palette("Paired", 12)
cos_cat_order = [
    "lncRNA only",
    "protein-coding only",
    "mixed",
]
cos_cat_colors = {
    "lncRNA only": _paired[5],
    "protein-coding only": _paired[1],
    "mixed": _paired[9],
}

share_biotypes = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"]
mod_order = ["lnc_eqtl", "lnc_sqtl", "pc_eqtl", "pc_sqtl"]
mod_label = {
    "lnc_eqtl": "lncRNA\neQTL",
    "lnc_sqtl": "lncRNA\nsQTL",
    "pc_eqtl": "protein-coding\neQTL",
    "pc_sqtl": "protein-coding\nsQTL",
}

np.random.seed(0)

# Zenodo ColocBoost helpers
def tissue_from_cos_id(cid):
    s = str(cid)
    i = s.rfind("_")
    return s[:i] if i > 0 else s


def lead_from_lists(variants, vcp):
    vs = list(variants)
    vcps = list(vcp)
    ps = pd.to_numeric(pd.Series(vcps), errors="coerce")
    if len(ps) == len(vs) and ps.notna().any():
        return vs[int(ps.values.argmax())]
    return vs[0]


def cos_to_long(row):
    vs = list(row["variants"])
    ps = list(row["vcp"])
    return pd.DataFrame({
        "variant_id": vs,
        "vcp": ps[: len(vs)] + [float("nan")] * (len(vs) - len(ps)),
    })


def _coloc_list_len(val):
    if val is None:
        return 0
    if isinstance(val, float) and np.isnan(val):
        return 0
    if isinstance(val, np.ndarray):
        return len(val)
    if isinstance(val, (list, tuple, set)):
        return len(val)
    return 0


def _typeset(vals):
    if vals is None:
        return set()
    if isinstance(vals, float) and np.isnan(vals):
        return set()
    if isinstance(vals, np.ndarray):
        items = vals.tolist()
    elif isinstance(vals, (list, tuple, set)):
        items = list(vals)
    else:
        items = [vals]
    out = set()
    for v in items:
        if v is None:
            continue
        s = str(v).strip()
        if s and s not in ("nan", "None", "<NA>"):
            out.add(s)
    return out


def _member_qtl_class(types):
    t = _typeset(types)
    lnc = t & {"novel_lncRNA", "existing_lncRNA"}
    non = t - {"novel_lncRNA", "existing_lncRNA"}
    if not lnc:
        return "non-lncRNA"
    if non:
        return "Shared"
    return "lncRNA-specific (robust)"


def _locus_qtl_label(member_classes):
    labs = set(member_classes)
    if labs == {"lncRNA-specific (robust)"}:
        return "lncRNA-specific (robust)"
    if "Shared" in labs or ("non-lncRNA" in labs and "lncRNA-specific (robust)" in labs):
        return "Shared"
    return "non-lncRNA"


def load_main_colocboost(colocboost_dir, *, cols=None):
    frames = []
    for path in sorted(colocboost_dir.glob("*.ColocBoost.parquet")):
        if path.name.startswith("cross_"):
            continue
        tissue = path.name.replace(".ColocBoost.parquet", "")
        df = pd.read_parquet(path, columns=cols)
        df["tissue"] = tissue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_cross_tissue_members(cross_tissue_loci_path, colocboost_dir, *, include_ucos=False):
    loci = pd.read_parquet(cross_tissue_loci_path)
    all_cos = load_main_colocboost(colocboost_dir).set_index("cos_id", drop=False)
    memb = loci.explode("member_cos_ids").rename(columns={"member_cos_ids": "cos_id"})
    memb["cos_id"] = memb["cos_id"].astype(str)
    cos_rows = memb[["locus_id", "cos_id"]].merge(
        all_cos.reset_index(drop=True), on="cos_id", how="inner"
    )
    cos_rows["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(cos_rows["variants"], cos_rows["vcp"])
    ]
    if "phenotype_id" not in cos_rows.columns:
        cos_rows["phenotype_id"] = cos_rows["phenotype_ids"]

    qtl_by_locus = (
        cos_rows.groupby("locus_id")["gene_types_robust"]
        .agg(lambda s: _locus_qtl_label(_member_qtl_class(x) for x in s))
    )
    locus_summary = loci.copy()
    locus_summary["qtl_label"] = locus_summary["locus_id"].map(qtl_by_locus)

    if not include_ucos:
        return locus_summary, cos_rows

    member_ids = set(memb["cos_id"])
    has_gwas = all_cos["gwas_traits"].map(_coloc_list_len) > 0
    has_qtl = all_cos["gene_ids_robust"].map(_coloc_list_len) > 0
    ucos = all_cos.loc[
        ~all_cos.index.isin(member_ids) & (has_gwas ^ has_qtl)
    ].reset_index(drop=True)
    ucos = ucos.loc[has_gwas.reindex(ucos["cos_id"]).fillna(False).values].copy()
    ucos["locus_id"] = "ucos_" + ucos["cos_id"].astype(str)
    ucos["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(ucos["variants"], ucos["vcp"])
    ]
    if "phenotype_id" not in ucos.columns:
        ucos["phenotype_id"] = ucos["phenotype_ids"]
    members = pd.concat([cos_rows, ucos], ignore_index=True)
    return locus_summary, members



## Paths

In [ ]:
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
gtex_colors_json = Path("refs/colors.json")
vep_or_file = Path("path/to/coloc_vep_odds.txt")

colocboost_dir = Path("path/to")
loci_path = Path("path/to/cross_tissue_loci.parquet")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_15_coloc_enrichment.pdf"
fig_png = fig_dir / "supp_figure_15_coloc_enrichment.png"


## Load data

### GENCODE gene classification

In [ ]:
gencode_genes = pd.read_table(gencode_class, low_memory=False)
gencode_genes = gencode_genes[gencode_genes["gene_type_grouped"] != "artifact"]
new_lncrna_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "novel_lncRNA", "gene_id_stable"])
existing_lncrna_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "existing_lncRNA", "gene_id_stable"])
protein_coding_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "protein_coding", "gene_id_stable"])
pseudogene_ids = set(gencode_genes.loc[gencode_genes["gene_type_grouped"] == "pseudogene", "gene_id_stable"])
lncrna_ids = new_lncrna_ids | existing_lncrna_ids
print(
    f"Novel lncRNAs: {len(new_lncrna_ids):,}; existing: {len(existing_lncrna_ids):,}; "
    f"p.c.: {len(protein_coding_ids):,}"
)

### ColocBoost cross-tissue loci (summary and members)

In [ ]:
locus_summary, locus_long = build_cross_tissue_members(loci_path, colocboost_dir, include_ucos=False)
locus_summary["n_gwas_gws"] = locus_summary["gwas_gws_traits"].map(len)
locus_summary["qtl_label_raw"] = locus_summary["qtl_label"].astype(str)
locus_summary["qtl_label"] = (
    locus_summary["qtl_label_raw"].map(qtl_label_for_figure).fillna(locus_summary["qtl_label_raw"])
)

print(
    f"Loci: {len(locus_summary):,}; members: {len(locus_long):,}; "
    f"GWS: {(locus_summary['n_gwas_gws'] > 0).sum():,}"
)
print("figure qtl_label (weak → Shared):")
print(locus_summary["qtl_label"].value_counts().to_string())
print("Note: analyses (including panel h) use CS-max GWAS p < 1e-5.")


### Single-tissue joint GWAS leads

In [ ]:
_lead_cols = [
    "cos_id",
    "gene_ids_robust",
    "phenotype_ids",
    "cos_max_pval_nominal_phenotypes",
    "variants",
    "vcp",
]
tissue_leads = load_main_colocboost(colocboost_dir, cols=_lead_cols)
tissue_leads["lead_variant_id"] = [
    lead_from_lists(v, p) for v, p in zip(tissue_leads["variants"], tissue_leads["vcp"])
]
print(
    f"Tissue ColocBoost CoS: {len(tissue_leads):,} rows across "
    f"{tissue_leads['tissue'].nunique()} tissues"
)


### GTEx tissue colors

In [ ]:
with open(gtex_colors_json) as _f:
    _colors = json.load(_f)
    _tc = pd.DataFrame.from_dict(_colors, orient="index").reset_index(names="tissue_name")
_tc["tissue_file_name"] = (
    _tc["tissue_name"]
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(" - ", "_")
    .str.replace(" ", "_")
)
_tc["tissue_color_hex"] = "#" + _tc["tissue_color_hex"].astype(str).str.lstrip("#")
tissue_color = dict(zip(_tc["tissue_file_name"], _tc["tissue_color_hex"]))
print(f"GTEx tissue colors: {len(tissue_color):,}")


### VEP annotation odds ratios for GWS CoS lead variants (panel h)

In [ ]:
# Full Firth OR table from variant_annotation/04_run_coloc_vcp_enrichment.py
# (all VEP annotations × CoS strata). Filter to the panel-h display set below.
coloc_annot_or_raw = pd.read_table(vep_or_file)
print(f"Loaded VEP ORs {vep_or_file.name}: {len(coloc_annot_or_raw):,} rows")
if "vcp_cutoff" in coloc_annot_or_raw.columns:
    coloc_annot_or_raw = coloc_annot_or_raw[coloc_annot_or_raw["vcp_cutoff"] == 0.9].copy()
    print(f"Filtered to vcp_cutoff==0.9: {len(coloc_annot_or_raw):,} rows")

# Display subset (notebook filter); keys are annotation column names from 03_/qtl_odds.
panel_h_annots = {
    "non_coding_transcript_exon_variant_lncRNA_d_v11": "lncRNA exon",
    "splicing_merged_lncRNA_d_v11": "lncRNA splicing",
    "intron_variant_lncRNA_d_v11": "lncRNA intron",
    "non_coding_transcript_exon_variant_protein_coding_d_v11": "p.c. exon",
    "splicing_merged_protein_coding_d_v11": "p.c. splicing",
    "intron_variant_protein_coding_d_v11": "p.c. intron",
    "missense_variant_protein_coding_d_v11": "missense",
    "synonymous_variant_protein_coding_d_v11": "synonymous",
    "3_prime_UTR_variant_protein_coding_d_v11": "3' UTR",
    "5_prime_UTR_variant_protein_coding_d_v11": "5' UTR",
    "stop_merged_protein_coding_d_v11": "stop / frameshift",
    "promoter_d_v11": "promoter",
    "enhancer_d_v11": "enhancer",
    "CTCF_binding_site_d_v11": "CTCF",
}

coloc_annot_or_df = coloc_annot_or_raw[
    coloc_annot_or_raw["annotation"].isin(panel_h_annots)
].copy()
coloc_annot_or_df["label"] = coloc_annot_or_df["annotation"].map(panel_h_annots)
# Plotting code expects OR / CI_low / CI_high / pvalue
coloc_annot_or_df["OR"] = coloc_annot_or_df["odds_ratio"]
coloc_annot_or_df["CI_low"] = coloc_annot_or_df["ci_lower"]
coloc_annot_or_df["CI_high"] = coloc_annot_or_df["ci_upper"]
coloc_annot_or_df["pvalue"] = coloc_annot_or_df["p_value"]
print(
    f"Panel h display rows: {len(coloc_annot_or_df):,} "
    f"({coloc_annot_or_df['label'].nunique()} annotations × "
    f"{coloc_annot_or_df['cos_cat'].nunique()} strata)"
)


## Run analysis

### Helpers and CoS composition labels

In [ ]:
def as_stem_set(vals):
    out = set()
    if vals is None:
        return out
    try:
        if isinstance(vals, float) and np.isnan(vals):
            return out
    except (TypeError, ValueError):
        pass
    items = vals if isinstance(vals, (list, tuple, np.ndarray, set)) else [vals]
    for v in items:
        if v is None:
            continue
        m = _ensg_stem.search(str(v))
        if m:
            out.add(m.group(1))
    return out


def parse_neglog10_list(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    out = []
    for tok in str(val).split(","):
        tok = tok.strip()
        if not tok or tok.lower() == "nan":
            out.append(np.nan)
        else:
            try:
                out.append(float(tok))
            except ValueError:
                out.append(np.nan)
    return out


def has_gwas_at_cut(phenotype_id, max_neg_log10_p, cut=gwas_neglog10p_1e5) -> bool:
    """True if any GWAS phenotype tip has CS-max -log10(p) > cut."""
    parts = [p.strip() for p in str(phenotype_id).split(";") if p.strip()]
    vals = parse_neglog10_list(max_neg_log10_p)
    for i, part in enumerate(parts):
        if i >= len(vals):
            break
        pl = part.lower()
        if "expression" in pl or "splicing" in pl:
            continue
        v = vals[i]
        if np.isfinite(v) and float(v) > cut:
            return True
    return False


def cos_composition(gene_stems):
    """Classify CoS by lncRNA / protein-coding gene counts (pseudo/sRNA ignored for category)."""
    n_lnc = sum(1 for g in gene_stems if g in lncrna_ids)
    n_pc = sum(1 for g in gene_stems if g in protein_coding_ids)
    if n_lnc >= 1 and n_pc >= 1:
        return "mixed"
    if n_lnc >= 1 and n_pc == 0:
        return "lncRNA only"
    if n_pc >= 1 and n_lnc == 0:
        return "protein-coding only"
    return None


def annotate_loci(df, name):
    out = df.copy()
    n_no_qtl = int((out["qtl_label"].astype(str) == "no_qtl").sum())
    if n_no_qtl:
        print(f"WARN: dropping {n_no_qtl:,} {name} loci with qtl_label=no_qtl")
        out = out[out["qtl_label"].astype(str) != "no_qtl"].copy()
    out["gene_stems"] = out["gene_ids_robust"].map(as_stem_set)
    out["n_genes"] = out["gene_stems"].map(len)
    out["cos_cat"] = out["gene_stems"].map(cos_composition)
    print(f"{name} gene-coloc loci: {len(out):,}")
    return out


# Drop not_sig (CS-max GWAS p ≥ 1e-5); keep sub_sig + gw_sig
_member_1e5 = [
    has_gwas_at_cut(pid, csmax, cut=gwas_neglog10p_1e5)
    for pid, csmax in zip(locus_long["phenotype_ids"], locus_long["cos_max_pval_nominal_phenotypes"])
]
locus_long_1e5 = locus_long[np.asarray(_member_1e5, dtype=bool)].copy()
loci_1e5_ids = set(locus_long_1e5["locus_id"].astype(str))
locus_summary_1e5 = locus_summary[
    locus_summary["locus_id"].astype(str).isin(loci_1e5_ids)
].copy()
print(
    f"p-tier filter: members all={len(locus_long):,} → p<1e-5={len(locus_long_1e5):,}; "
    f"loci all={len(locus_summary):,} → p<1e-5={len(locus_summary_1e5):,}; "
    f"GWS={int((locus_summary['n_gwas_gws'] > 0).sum()):,}"
)

all_loci = annotate_loci(locus_summary_1e5, "p<1e-5")
gws_loci = annotate_loci(
    locus_summary[locus_summary["n_gwas_gws"] > 0], "GWS"
)
print("GWS CoS composition:")
print(gws_loci["cos_cat"].value_counts(dropna=False).to_string())

### QTL phenotype counts (panels d–g)

In [ ]:
def qtl_pheno_records(phenotype_id):
    """Tissue-collapsed QTL phenotypes: (key, modality class or None)."""
    recs = []
    for part in str(phenotype_id).split(";"):
        part = part.strip()
        if not part:
            continue
        pl = part.lower()
        if "expression" in pl:
            kind = "eqtl"
        elif "splicing" in pl:
            kind = "sqtl"
        else:
            continue
        body = part.split(".", 1)[1] if "." in part else part
        body = re.sub(r"\.\d+$", "", body)
        m = _ensg_stem.search(body)
        stem = m.group(1) if m else None
        cls = None
        if stem in lncrna_ids and kind == "eqtl":
            cls = "lnc_eqtl"
        elif stem in lncrna_ids and kind == "sqtl":
            cls = "lnc_sqtl"
        elif stem in protein_coding_ids and kind == "eqtl":
            cls = "pc_eqtl"
        elif stem in protein_coding_ids and kind == "sqtl":
            cls = "pc_sqtl"
        recs.append((body, cls, stem))
    return recs


keys_by_locus = {}
for locus_id, phenotype_id in zip(locus_long_1e5["locus_id"], locus_long_1e5["phenotype_ids"]):
    lid = str(locus_id)
    keys = keys_by_locus.setdefault(lid, set())
    for key, cls, stem in qtl_pheno_records(phenotype_id):
        keys.add(key)

locus_summary_1e5 = locus_summary_1e5.copy()
locus_summary_1e5["n_qtl_phenotypes"] = (
    locus_summary_1e5["locus_id"].astype(str).map(lambda i: len(keys_by_locus.get(i, ())))
)
# cross_tissue_loci.parquet on Zenodo has no n_tissues / n_gwas_traits / n_genes.
_qtl_memb = locus_long_1e5[
    locus_long_1e5["gene_ids_robust"].map(lambda x: x is not None and len(x) > 0)
]
_n_tissues = (
    _qtl_memb.groupby(_qtl_memb["locus_id"].astype(str))["tissue"].nunique()
)
locus_summary_1e5["n_tissues"] = (
    locus_summary_1e5["locus_id"].astype(str).map(_n_tissues).fillna(0).astype(int)
)
locus_summary_1e5["n_gwas_traits"] = locus_summary_1e5["gwas_traits"].map(len)
locus_summary_1e5["n_genes"] = locus_summary_1e5["gene_ids_robust"].map(as_stem_set).map(len)
print(
    f"QTL phenotypes per p<1e-5 CoS: median {locus_summary_1e5['n_qtl_phenotypes'].median():.0f}, "
    f"max {int(locus_summary_1e5['n_qtl_phenotypes'].max())}; "
    f"n_tissues median {locus_summary_1e5['n_tissues'].median():.0f}"
)


def _same_biotype(a, b):
    return a.split("_")[0] == b.split("_")[0]

### Co-occurrence matrix and % multi-gene by biotype (panels a–b)

In [ ]:
def pct_multigene_cos(loci_df, scope):
    """Among GWAS-QTL CoS with ≥1 gene of the biotype, % with ≥2 genes."""
    id_map = {
        "novel_lncRNA": new_lncrna_ids,
        "existing_lncRNA": existing_lncrna_ids,
        "protein_coding": protein_coding_ids,
        "pseudogene": pseudogene_ids,
    }
    rows = []
    for bt in share_biotypes:
        ids = id_map[bt]
        n_cos = 0
        n_multi = 0
        for stems in loci_df["gene_stems"]:
            if not (stems & ids):
                continue
            n_cos += 1
            if len(stems) > 1:
                n_multi += 1
        pct = 100.0 * n_multi / n_cos if n_cos else np.nan
        rows.append({
            "scope": scope,
            "biotype": bt,
            "n_cos": n_cos,
            "n_cos_multigene": n_multi,
            "pct_multigene": pct,
        })
    return pd.DataFrame(rows)


st_loci = tissue_leads[
    (tissue_leads["gene_ids_robust"].map(len) >= 1)
].copy()
st_loci = st_loci[
    [
        has_gwas_at_cut(pid, csmax, cut=gwas_neglog10p_1e5)
        for pid, csmax in zip(st_loci["phenotype_ids"], st_loci["cos_max_pval_nominal_phenotypes"])
    ]
].copy()
st_loci["gene_stems"] = st_loci["gene_ids_robust"].map(as_stem_set)
st_loci["n_genes"] = st_loci["gene_stems"].map(len)
st_loci = st_loci[st_loci["n_genes"] >= 1].copy()
print(
    f"Single-tissue p<1e-5 gene-coloc CoS: {len(st_loci):,} "
    f"across {st_loci['tissue'].nunique()} tissues"
)

share_pct_st = pct_multigene_cos(st_loci, "single_tissue")
share_pct_ct = pct_multigene_cos(all_loci, "cross_tissue")
share_pct_df = pd.concat([share_pct_st, share_pct_ct], ignore_index=True)

_tissue_share_rows = []
for _t, _g in st_loci.groupby("tissue", sort=True):
    _sub = pct_multigene_cos(_g, "single_tissue")
    _sub["tissue"] = _t
    _tissue_share_rows.append(_sub)
share_pct_by_tissue = pd.concat(_tissue_share_rows, ignore_index=True)
share_pct_by_tissue = share_pct_by_tissue[share_pct_by_tissue["n_cos"] > 0].copy()

print("% of GWAS-QTL CoS that are multi-gene, among CoS with that biotype:")
print(share_pct_df.to_string(index=False))

# Panel a co-occurrence counts (single-tissue CoS concatenated across tissues)
n_mod = len(mod_order)
alone_n = np.zeros(n_mod, dtype=int)
other_n = np.zeros(n_mod, dtype=int)
same_n = np.zeros((n_mod, n_mod), dtype=int)
diff_n = np.zeros((n_mod, n_mod), dtype=int)
row_n = np.zeros(n_mod, dtype=int)
for phenotype_id in tqdm(st_loci["phenotype_ids"], desc="panel a"):
    bodies = {k: {} for k in mod_order}
    all_stems = set()
    for key, cls, stem in qtl_pheno_records(phenotype_id):
        if not stem:
            continue
        all_stems.add(stem)
        if cls:
            bodies[cls].setdefault(stem, set()).add(key)
    if not all_stems:
        continue
    focal = {k: set(bodies[k]) for k in mod_order}
    other_stems = all_stems - set().union(*focal.values())
    for i, row in enumerate(mod_order):
        genes = focal[row]
        if not genes:
            continue
        row_n[i] += 1
        only_this = not other_stems and not any(focal[k] for k in mod_order if k != row)
        multi_self = row.endswith("sqtl") and any(len(bodies[row][s]) >= 2 for s in genes)
        if len(all_stems) == 1 and len(genes) == 1 and only_this and not multi_self:
            alone_n[i] += 1
            continue
        for j, col in enumerate(mod_order):
            partner = focal[col]
            if _same_biotype(row, col):
                if row == col and col.endswith("eqtl"):
                    if len(genes) >= 2:
                        diff_n[i, j] += 1
                elif row == col:
                    if multi_self:
                        same_n[i, j] += 1
                    if len(partner) >= 2:
                        diff_n[i, j] += 1
                else:
                    if genes & partner:
                        same_n[i, j] += 1
                    if partner - genes:
                        diff_n[i, j] += 1
            elif partner:
                diff_n[i, j] += 1
        if other_stems:
            other_n[i] += 1


def _row_pct(num):
    out = np.zeros_like(num, dtype=float)
    denom = row_n.reshape((-1,) + (1,) * (num.ndim - 1))
    np.divide(100.0 * num, denom, out=out, where=denom > 0)
    return out


alone_pct = _row_pct(alone_n)
other_pct = _row_pct(other_n)
same_pct = _row_pct(same_n)
diff_pct = _row_pct(diff_n)
print("Panel a, % of single-tissue CoS with the row class (columns overlap):")
for i, row in enumerate(mod_order):
    total = alone_pct[i] + other_pct[i]
    for j in range(n_mod):
        total += same_pct[i, j] + diff_pct[i, j]
    print(f"  {row}: n={int(row_n[i]):,}  column sum={total:.1f}")


### Single-tissue → cross-tissue flow (panel c)

In [ ]:
ct_n_genes = {
    str(lid): len(stems)
    for lid, stems in zip(all_loci["locus_id"], all_loci["gene_stems"])
}
flow_groups = {
    "protein_coding": protein_coding_ids,
    "lncRNA": lncrna_ids,
}
flow_rows = []
for lid, gene_ids in zip(locus_long_1e5["locus_id"], locus_long_1e5["gene_ids_robust"]):
    stems = as_stem_set(gene_ids)
    if not stems:
        continue
    left = "single-gene" if len(stems) == 1 else "multi-gene"
    right = "shared" if ct_n_genes.get(str(lid), 0) > 1 else "single-gene"
    for group, ids in flow_groups.items():
        if stems & ids:
            flow_rows.append({"group": group, "left": left, "right": right})
sankey_df = (
    pd.DataFrame(flow_rows)
    .value_counts()
    .rename("n_cos")
    .reset_index()
)
for group in flow_groups:
    sub = sankey_df[sankey_df["group"] == group].copy()
    n = int(sub["n_cos"].sum())
    print(f"{group}: {n:,} single-tissue CoS")
    print(sub.assign(pct=100.0 * sub["n_cos"] / n).to_string(index=False))

## Make figure

In [ ]:
def stars(p):
    if p is None or not np.isfinite(p):
        return ""
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return ""


fig = plt.figure(figsize=(7.2, 5.0), dpi=300)
gs = fig.add_gridspec(3, 1, height_ratios=[1.30, 0.68, 1.30], hspace=0.78)

# Top: a co-occurrence, b multi-gene, c single-tissue vs cross-tissue
gs_top = gs[0].subgridspec(
    1, 5, width_ratios=[1.70, 0.92, 1.18, 0.28, 1.35], wspace=0,
)
gs_e = gs_top[0, 0].subgridspec(
    1, 2, width_ratios=[1, 0.028], wspace=0.04,
)
ax_hm = fig.add_subplot(gs_e[0, 0])
cax = fig.add_subplot(gs_e[0, 1].subgridspec(3, 1, height_ratios=[0.25, 0.5, 0.25], hspace=0)[1, 0])
ax_f = fig.add_subplot(gs_top[0, 2])
gs_flow = gs_top[0, 4].subgridspec(1, 2, wspace=0.28)
ax_g_pc = fig.add_subplot(gs_flow[0, 0])
ax_g_lnc = fig.add_subplot(gs_flow[0, 1])

# Middle: d–g — locus-structure histograms. g is clipped at 100.
gs_mid = gs[1].subgridspec(1, 4, wspace=0.55)
axes_top = [fig.add_subplot(gs_mid[0, 0])]
axes_top += [fig.add_subplot(gs_mid[0, i], sharey=axes_top[0]) for i in range(1, 4)]
ax_h = fig.add_subplot(gs[2])

xlabels_top = [
    "Tissues w/ QTL in CoS",
    "GWAS in CoS",
    "Genes w/ QTL in CoS",
    "QTL phenotypes in CoS",
]
for ax, col, xlab in zip(
    axes_top,
    ["n_tissues", "n_gwas_traits", "n_genes", "n_qtl_phenotypes"],
    xlabels_top,
):
    if col == "n_qtl_phenotypes":
        xplot = locus_summary_1e5[col].clip(upper=100)
        sns.histplot(x=xplot, discrete=True, ax=ax, color="#bdbdbd", edgecolor="none")
        ax.set_xlim(0.5, 100.5)
        ax.set_xticks([20, 40, 60, 80, 100])
        ax.set_xticklabels(["20", "40", "60", "80", r"$\geq$100"])
    else:
        sns.histplot(locus_summary_1e5, x=col, discrete=True, ax=ax, color="#bdbdbd", edgecolor="none")
    ax.set_xlabel(xlab)
    ax.set_yscale("log")
    ax.set_ylabel("cross-tissue\nGWAS-QTL CoS")
    ax.tick_params(labelleft=True)
    sns.despine(ax=ax)

# Gaps between alone | each QTL class | other gene. Each class is same gene, then diff. gene.
gap = 0.45
group_spec = [
    ("alone", 1, "alone"),
    ("lnc_eqtl", 2, "lncRNA\neQTL"),
    ("lnc_sqtl", 2, "lncRNA\nsQTL"),
    ("pc_eqtl", 2, "p.c.\neQTL"),
    ("pc_sqtl", 2, "p.c.\nsQTL"),
    ("other", 1, "other\ngene"),
]
x_at = {}
x_cursor = 0.0
for name, width, _lab in group_spec:
    x_at[name] = x_cursor
    x_cursor += width + gap
xmax = x_cursor - gap

pieces = []
for i, row in enumerate(mod_order):
    pieces.append((i, x_at["alone"], 1, float(alone_pct[i])))
    for j, col in enumerate(mod_order):
        x0 = x_at[col]
        grey_same = (row == col and col.endswith("eqtl")) or not _same_biotype(row, col)
        pieces.append((i, x0, 1, None if grey_same else float(same_pct[i, j])))
        pieces.append((i, x0 + 1, 1, float(diff_pct[i, j])))
    pieces.append((i, x_at["other"], 1, float(other_pct[i])))
norm = plt.Normalize(vmin=0, vmax=100)
cmap = plt.get_cmap("Blues")
for i, x, w, val in pieces:
    if val is None:
        ax_hm.add_patch(Rectangle(
            (x, i - 0.5), w, 1, facecolor="#d9d9d9",
            edgecolor="white", linewidth=0.8, zorder=2,
        ))
        continue
    ax_hm.add_patch(Rectangle(
        (x, i - 0.5), w, 1, facecolor=cmap(norm(val)),
        edgecolor="white", linewidth=0.8, zorder=2,
    ))
# Diff. gene and other gene are a second gene. Alone and same gene are one gene.
_inset = 0.07
for col in mod_order:
    ax_hm.add_patch(Rectangle(
        (x_at[col] + 1 + _inset, -0.5 + _inset),
        1 - 2 * _inset, len(mod_order) - 2 * _inset,
        facecolor="none", edgecolor="k", linewidth=0.9, zorder=3,
    ))
ax_hm.add_patch(Rectangle(
    (x_at["other"] + _inset, -0.5 + _inset),
    1 - 2 * _inset, len(mod_order) - 2 * _inset,
    facecolor="none", edgecolor="k", linewidth=0.9, zorder=3,
))
for i, x, w, val in pieces:
    if val is None:
        continue
    label = f"{val:.0f}" if val >= 10 else f"{val:.1f}"
    ax_hm.text(
        x + w / 2, i, label, ha="center", va="center", fontsize=2.8, zorder=10,
        color="white" if val > 55 else "black",
        clip_on=True,
    )
ax_hm.set_xlim(-0.02, xmax + 0.02)
ax_hm.set_ylim(len(mod_order) - 0.5, -0.5)
ax_hm.set_xticks([])
ax_hm.set_yticks(range(len(mod_order)))
ax_hm.set_yticklabels([mod_label[k] for k in mod_order], fontsize=4.2, ha="right")
ax_hm.tick_params(axis="y", length=0, pad=0.2)
for sp in ax_hm.spines.values():
    sp.set_visible(False)

for name, width, lab in group_spec:
    x0 = x_at[name]
    if width == 2:
        ax_hm.text(
            x0 + 0.5, 1.01, "same\ngene", ha="center", va="bottom", fontsize=2.6,
            clip_on=False, transform=ax_hm.get_xaxis_transform(),
        )
        ax_hm.text(
            x0 + 1.5, 1.01, "diff.\ngene", ha="center", va="bottom", fontsize=2.6,
            clip_on=False, transform=ax_hm.get_xaxis_transform(),
        )
        bracket_y = 1.14
        lab_y = 1.18
    else:
        bracket_y = 1.06
        lab_y = 1.10
    ax_hm.plot(
        [x0 + 0.06, x0 + width - 0.06], [bracket_y, bracket_y],
        color="0.2", lw=0.7, solid_capstyle="butt", clip_on=False,
        transform=ax_hm.get_xaxis_transform(),
    )
    ax_hm.text(
        x0 + width / 2, lab_y, lab, ha="center", va="bottom", fontsize=3.1,
        clip_on=False, transform=ax_hm.get_xaxis_transform(),
    )
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_ticks([0, 50, 100])
cbar.ax.tick_params(labelsize=3.5, length=2)
cbar.set_label("% of single-tissue\nGWAS-QTL CoS", fontsize=3.3, labelpad=1.2)

# Panel b — per-tissue dots + cross-tissue outline bar
biotype_ticks = {
    "novel_lncRNA": "Novel\nlncRNA",
    "existing_lncRNA": "Existing\nlncRNA",
    "protein_coding": "Protein-\ncoding",
    "pseudogene": "Pseudogene",
}
share_ct_idx = share_pct_ct.set_index("biotype")
n_f = len(share_biotypes)
x_f = {bt: i for i, bt in enumerate(share_biotypes)}
bar_w = 0.72
for bt in share_biotypes:
    ct_v = float(share_ct_idx.loc[bt, "pct_multigene"])
    ax_f.bar(
        x_f[bt], ct_v, width=bar_w,
        facecolor="none", edgecolor=gene_type_palette[bt],
        linewidth=0.95, zorder=1,
    )

_dot = share_pct_by_tissue.copy()
_dot["x"] = _dot["biotype"].map(x_f).astype(float)
rng = np.random.default_rng(0)
_dot["x_jit"] = _dot["x"] + rng.uniform(-0.18, 0.18, size=len(_dot))
ax_f.scatter(
    _dot["x_jit"], _dot["pct_multigene"],
    c=_dot["tissue"].map(tissue_color).fillna("0.45"),
    s=4.5, linewidths=0, alpha=0.9, zorder=3,
)
ax_f.set_xticks([x_f[bt] for bt in share_biotypes])
ax_f.set_xticklabels([biotype_ticks[bt] for bt in share_biotypes], fontsize=3.4)
ax_f.tick_params(axis="x", pad=1)
ax_f.set_ylabel("% multi-gene\nGWAS-QTL CoS")
ax_f.set_ylim(0, 100)
ax_f.set_xlim(-0.55, n_f - 0.45)
ax_f.spines["top"].set_visible(False)
ax_f.spines["right"].set_visible(False)
ax_f.legend(
    handles=[
        Patch(facecolor="none", edgecolor="0.35", linewidth=0.9, label="cross-tissue"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="0.35",
               markersize=2.8, label="single tissue"),
    ],
    loc="lower left", frameon=True, fancybox=False,
    facecolor="white", edgecolor="0.75", framealpha=1.0,
    fontsize=3.6, handletextpad=0.4, labelspacing=0.35, borderpad=0.5,
)

# Panel c — protein-coding and lncRNA, single tissue → cross tissue
left_order = ["multi-gene", "single-gene"]
right_order = ["shared", "single-gene"]
flow_color = {
    ("multi-gene", "shared"): "#bdbdbd",
    ("single-gene", "shared"): "#d6d6d6",
    ("single-gene", "single-gene"): "#e8e8e8",
    ("multi-gene", "single-gene"): "#bdbdbd",
}
node_face = {"multi-gene": "#e6e6e6", "shared": "#e6e6e6", "single-gene": "#f5f5f5"}
node_ink = {"multi-gene": "black", "shared": "black", "single-gene": "black"}
node_gap = 4.0

for ax, group, title, show_headers in (
    (ax_g_pc, "protein_coding", "protein-coding", True),
    (ax_g_lnc, "lncRNA", "lncRNA", True),
):
    flow_df = sankey_df[sankey_df["group"] == group]
    n_tot = float(flow_df["n_cos"].sum())
    left_pct = {
        k: 100.0 * float(flow_df.loc[flow_df["left"] == k, "n_cos"].sum()) / n_tot
        for k in left_order
    }
    right_pct = {
        k: 100.0 * float(flow_df.loc[flow_df["right"] == k, "n_cos"].sum()) / n_tot
        for k in right_order
    }
    flow_pct = {
        (L, R): (
            100.0
            * float(
                flow_df.loc[
                    (flow_df["left"] == L) & (flow_df["right"] == R), "n_cos"
                ].sum()
            )
            / n_tot
        )
        for L in left_order
        for R in right_order
    }

    y = sum(left_pct[k] for k in left_order) + node_gap * (len(left_order) - 1)
    y_top = y
    left_span = {}
    for k in left_order:
        h = left_pct[k]
        left_span[k] = [y - h, y]
        y -= h + node_gap

    y = sum(right_pct[k] for k in right_order) + node_gap * (len(right_order) - 1)
    right_span = {}
    for k in right_order:
        h = right_pct[k]
        right_span[k] = [y - h, y]
        y -= h + node_gap

    left_cursor = {k: left_span[k][1] for k in left_order}
    right_cursor = {k: right_span[k][1] for k in right_order}
    for left_name in left_order:
        for right_name in right_order:
            h = flow_pct[(left_name, right_name)]
            if h <= 0:
                continue
            y0_top = left_cursor[left_name]
            y0_bot = y0_top - h
            left_cursor[left_name] = y0_bot
            y1_top = right_cursor[right_name]
            y1_bot = y1_top - h
            right_cursor[right_name] = y1_bot
            t = np.linspace(0, 1, 40)
            s = t * t * (3 - 2 * t)
            x = 0.20 + 0.60 * t
            top_y = y0_top + (y1_top - y0_top) * s
            bot_y = y0_bot + (y1_bot - y0_bot) * s
            ax.fill(
                np.concatenate([x, x[::-1]]),
                np.concatenate([top_y, bot_y[::-1]]),
                facecolor=flow_color[(left_name, right_name)],
                edgecolor="none",
                alpha=0.4,
                zorder=1,
            )
    for name, (bot, top) in left_span.items():
        ax.add_patch(Rectangle(
            (0.02, bot), 0.18, top - bot,
            facecolor=node_face[name], edgecolor="k", linewidth=0.4, zorder=2,
        ))
        ax.text(
            0.11, (bot + top) / 2, f"{name}\n{left_pct[name]:.0f}%",
            ha="center", va="center", fontsize=3.2, color=node_ink[name], zorder=3,
        )
    for name, (bot, top) in right_span.items():
        ax.add_patch(Rectangle(
            (0.80, bot), 0.18, top - bot,
            facecolor=node_face[name], edgecolor="k", linewidth=0.4, zorder=2,
        ))
        ax.text(
            0.89, (bot + top) / 2, f"{name}\n{right_pct[name]:.0f}%",
            ha="center", va="center", fontsize=3.2, color=node_ink[name], zorder=3,
        )
    ax.text(
        0.50, 1.14, title,
        ha="center", va="bottom", fontsize=4.2, color="black",
        transform=ax.transAxes, clip_on=False,
    )
    if show_headers:
        ax.text(
            0.11, 1.02, "single\ntissue", ha="center", va="bottom", fontsize=3.2,
            transform=ax.transAxes, clip_on=False,
        )
        ax.text(
            0.89, 1.02, "cross\ntissue", ha="center", va="bottom", fontsize=3.2,
            transform=ax.transAxes, clip_on=False,
        )
    ax.set_xlim(0, 1)
    ax.set_ylim(-1, y_top + 1)
    ax.axis("off")

# Panel h — VEP lead enrichment by CoS composition
plot_f = coloc_annot_or_df[coloc_annot_or_df["cos_cat"].isin(cos_cat_order)].copy()

# Sort annotations by lncRNA-only OR (highest enrichment at left)
sort_cat = "lncRNA only"
lnc_or = (
    plot_f.loc[plot_f["cos_cat"] == sort_cat, ["label", "OR"]]
    .drop_duplicates("label")
    .set_index("label")["OR"]
)
annot_order = list(lnc_or.sort_values(ascending=False).index)
missing = [lab for lab in plot_f["label"].unique() if lab not in annot_order]
if missing:
    med = (
        plot_f[plot_f["label"].isin(missing)]
        .groupby("label")["OR"].median()
        .sort_values(ascending=False)
    )
    annot_order.extend(list(med.index))

plot_f["label"] = pd.Categorical(plot_f["label"], categories=annot_order, ordered=True)
plot_f = plot_f.dropna(subset=["label"]).sort_values(["label", "cos_cat"])

x_base = {lab: i for i, lab in enumerate(annot_order)}
n_cat = len(cos_cat_order)
offsets = {
    lab: -0.28 + i * (0.56 / max(n_cat - 1, 1))
    for i, lab in enumerate(cos_cat_order)
}

# Track max CI top (incl. stars) for ylim headroom above legend
or_hi_max = 1.0
plot_idx = plot_f.set_index(["cos_cat", "label"])
for lab in cos_cat_order:
    for annot in annot_order:
        x = x_base[annot] + offsets[lab]
        if (lab, annot) not in plot_idx.index:
            ax_h.plot(x, 1.0, marker="x", color="0.55", ms=3.5, mew=0.9, zorder=3)
            continue
        row = plot_idx.loc[(lab, annot)]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        or_hat, lo, hi = row["OR"], row["CI_low"], row["CI_high"]
        # NaN / degenerate (0 or ∞ CI) — grey x
        if not (np.isfinite(or_hat) and np.isfinite(lo) and np.isfinite(hi) and or_hat > 0):
            ax_h.plot(x, 1.0, marker="x", color="0.55", ms=3.5, mew=0.9, zorder=3)
            continue
        ax_h.errorbar(
            x, or_hat,
            yerr=[[or_hat - lo], [hi - or_hat]],
            fmt="o", color=cos_cat_colors[lab],
            ecolor=cos_cat_colors[lab], elinewidth=0.75, capsize=0,
            markersize=3.0, markeredgecolor="k", markeredgewidth=0.25,
            zorder=3,
        )
        star = stars(row["pvalue"])
        if star:
            ax_h.text(
                x, hi * 1.15, star,
                va="bottom", ha="center", fontsize=3.2, color="black",
                clip_on=False, zorder=4,
            )
            or_hi_max = max(or_hi_max, float(hi) * 1.15)
        else:
            or_hi_max = max(or_hi_max, float(hi))

ax_h.axhline(1.0, color="grey", ls="--", lw=0.8, zorder=0)
for _x in range(1, len(annot_order)):
    ax_h.axvline(_x - 0.5, color="0.75", ls="-", lw=0.5, zorder=0)
ax_h.set_xticks(list(x_base.values()))
ax_h.set_xticklabels(list(x_base.keys()), rotation=40, ha="right")
ax_h.set_ylabel("Odds ratio")
ax_h.set_yscale("log", base=2)
ylim_top = max(32.0, or_hi_max * 1.35)
ax_h.set_ylim(0.25, ylim_top)
ax_h.set_xlim(-0.6, len(annot_order) - 0.4)
ax_h.spines["top"].set_visible(False)
ax_h.spines["right"].set_visible(False)
ax_h.legend(
    handles=[
        Line2D(
            [], [], marker="o", ls="",
            markerfacecolor=cos_cat_colors[lab],
            markeredgecolor="k", markeredgewidth=0.25, ms=2.8, label=lab,
        )
        for lab in cos_cat_order
    ],
    loc="lower center", bbox_to_anchor=(0.5, .93),
    frameon=False, title="Cross-tissue GWAS-QTL CoS composition",
    fontsize=4.2, title_fontsize=4.2, ncol=3,
    handletextpad=0.4, columnspacing=1.0,
)

fig.subplots_adjust(bottom=0.14, top=0.88)
fig.canvas.draw()
_renderer = fig.canvas.get_renderer()
_inv = fig.transFigure.inverted()

def _left_of_ylabel(ax, fallback_dx=-0.55):
    lab = ax.yaxis.label
    if lab.get_text():
        bb = lab.get_window_extent(renderer=_renderer).transformed(_inv)
        return bb.x0 - 0.010
    return _inv.transform(ax.transAxes.transform((fallback_dx, 1.0)))[0]

_letter_y_abc = max(ax_hm.get_position().y1, ax_f.get_position().y1, ax_g_pc.get_position().y1) + 0.048
_letter_y_d = axes_top[0].get_position().y1 + 0.014
_letter_y_h = ax_h.get_position().y1 + 0.012
# a: left of heatmap y tick labels
_yt = ax_hm.get_yticklabels()
_a_bb = _yt[0].get_window_extent(renderer=_renderer).transformed(_inv) if _yt else None
_letter_x_a = (_a_bb.x0 - 0.010) if _a_bb is not None else _inv.transform(ax_hm.transAxes.transform((-0.08, 1.0)))[0]
_letter_x_b = _left_of_ylabel(ax_f, fallback_dx=-0.70)
_letter_x_c = _inv.transform(ax_g_pc.transAxes.transform((-0.14, 1.0)))[0]
_letter_x_h = _left_of_ylabel(ax_h, fallback_dx=-0.20)
for ax, letter, x_fig, y_fig in (
    (ax_hm, "a", _letter_x_a, _letter_y_abc),
    (ax_f, "b", _letter_x_b, _letter_y_abc),
    (ax_g_pc, "c", _letter_x_c, _letter_y_abc),
    (axes_top[0], "d", None, _letter_y_d),
    (axes_top[1], "e", None, _letter_y_d),
    (axes_top[2], "f", None, _letter_y_d),
    (axes_top[3], "g", None, _letter_y_d),
    (ax_h, "h", _letter_x_h, _letter_y_h),
):
    if x_fig is None:
        x_fig = _left_of_ylabel(ax, fallback_dx=-0.78)
    fig.text(x_fig, y_fig, letter, fontweight="bold", va="bottom", ha="right", fontsize=9)
plt.show()


## Save figure

In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")

## Print stats

In [ ]:
# Panel c: % of single-tissue CoS with multiple genes, by biotype
for group, label in [("lncRNA", "lncRNA"), ("protein_coding", "protein-coding")]:
    sub = sankey_df[sankey_df["group"] == group]
    pct_multi = 100.0 * sub.loc[sub["left"] == "multi-gene", "n_cos"].sum() / sub["n_cos"].sum()
    print(f"Multi-gene colocalizations, {label}: {pct_multi:.0f}%")

# Panel h: high-VCP (max VCP>0.9) variants by exact CoS composition (p<1e-5)
vep_strata_n = coloc_annot_or_df.groupby("cos_cat")["n_case"].max()
for lab in cos_cat_order:
    print(f"VEP stratum n, {lab}: {vep_strata_n[lab]:,}")
